In [ ]:
from datasets import load_dataset
from dotenv import find_dotenv, load_dotenv
from pinecone import Pinecone, ServerlessSpec
import os
from sentence_transformers import SentenceTransformer

load_dotenv(find_dotenv(), override=True)

In [ ]:
fw = load_dataset(
    "HuggingFaceFW/fineweb",
    name="sample-10BT",
    split="train",
    streaming=True,
    token=os.getenv("HF_TOKEN")
)

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")

In [ ]:
load_dotenv(find_dotenv() , override=True)
pc = Pinecone(
    api_key=os.getenv("PINECONE_API_KEY")
)

In [ ]:
name_text = "text_01"

existing_indexes = [index.name for index in pc.list_indexes()]

if name_text not in existing_indexes:
    pc.create_index(
        name=name_text,
        dimension=model.get_sentence_embedding_dimension(),
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

In [ ]:
index = pc.Index(name=name_text)

In [ ]:
sub_set = 1000

vectors_to_upsert = []

for i, item in enumerate(fw):

    if i >= sub_set:
        break

    text = item["text"]
    unique_id = item["id"]
    language = item["language"]

    # Create embedding
    embedding = model.encode(
        text,
        show_progress_bar=False
    ).tolist()

    metadata = {
        "language": language,
        "text": text
    }

    vectors_to_upsert.append(
        (unique_id, embedding, metadata)
    )

print(f"Prepared {len(vectors_to_upsert)} vectors")